In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType,
    TimestampType,
    FloatType,
)
import pyspark.sql.functions as F
from  pyspark.sql import Row
catalog_name = 'ecommerce'

In [0]:
df_product=spark.read.table(f'{catalog_name}.silver.slv_product')
df_category=spark.read.table(f'{catalog_name}.silver.slv_category')
df_brands=spark.read.table(f'{catalog_name}.silver.slv_brands')

df_product.show()

In [0]:
df_category.show()

In [0]:
df_brands.show()


In [0]:
df_product.createOrReplaceTempView("view_product")
df_category.createOrReplaceTempView("view_category")
df_brands.createOrReplaceTempView("view_brand")

In [0]:
%sql
select * from view_product;

In [0]:
%sql
use catalog ecommerce;

In [0]:
%sql
    select b.brand_code,b.brand_name,c.category_code,c.category_name from view_brand b join view_category c on b.category_code = c.category_code

In [0]:
%sql
create or replace table gold.gld_product as
with brand_categories as ( --cte 
  select
    b.brand_code,
    b.brand_name,
    c.category_code,
    c.category_name
  from
    view_brand b
      join view_category c
        on b.category_code = c.category_code
)
select
  p.product_id,
  p.sku,
  p.category_code,
  coalesce(bc.category_name, "Not Available") as category_name,
  p.brand_code,
  coalesce(bc.brand_name, "Not Available") as brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p.file_name,
  p.ingest_timestamp

from
  view_product p
    left join brand_categories bc
      on p.brand_code = bc.brand_code

In [0]:
x=df_product.join(df_brands,how="inner",on="brand_code").join(df_category,how="inner",on="category_code")
x.show()
# x=x.join(df_category,how="inner",on="category_code")
# x.show()